1. Model: Qwen3-VL-8B vision with Unsloth (4-bit)

In [ ]:
!pip install -U "unsloth>=2024.10.0" "transformers>=4.57.0" datasets pillow accelerate bitsandbytes trl

2. Dataset: use your JSONL + real images

You already have JSONL rows like (simplified):

In [10]:
from datasets import load_dataset
from PIL import Image
import os

DATA_PATH   = "training_dataset.jsonl"   # your JSONL
IMAGES_ROOT = "."                           # root so that url is relative, change if needed

dataset = load_dataset(
    "json",
    data_files={"train": DATA_PATH},
)["train"]


Convert messages into Unsloth vision format

In [11]:
# Convert HF Dataset → Python list (required by SFTTrainer for vision)
train_data = dataset.to_list()

print("Loaded examples:", len(train_data))


Loaded examples: 221


In [13]:
print(train_data[0]["messages"][1]["content"])
# ex = train_data[0]
# for msg in ex["messages"]:
#     print("ROLE:", msg["role"])
#     for block in msg["content"]:
#         print("   ", block["type"], type(block.get("image") or block.get("text")))


[{'type': 'text', 'text': 'Extract all fields from this referral form and return your answer as a JSON object.', 'image': None}, {'type': 'image', 'text': None, 'image': 'images/gp-referral-cancer-colorectal IOV108_page_1.png'}]


In [15]:
def convert_for_vision(example):
    new_messages = []

    for msg in example["messages"]:
        role = msg["role"]
        blocks = []

        for block in msg["content"]:
            b_type = block.get("type")

            # Keep text blocks unchanged
            if b_type == "text":
                blocks.append({
                    "type": "text",
                    "text": block["text"],
                })

            # Keep image blocks unchanged (must be a string path)
            elif b_type == "image":
                blocks.append({
                    "type": "image",
                    "image": block["image"],  # string path only
                })

            # Any unexpected block becomes text so template doesn’t break
            else:
                blocks.append({
                    "type": "text",
                    "text": str(block),
                })

        new_messages.append({
            "role": role,
            "content": blocks,
        })

    return {"messages": new_messages}

converted_ds = [convert_for_vision(ex) for ex in train_data]

In [18]:
print(converted_ds[0])

{'messages': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a referral form assistant. Extract a structured JSON response following the known schema. Include both the extracted values and their bounding boxes. Leave missing fields as null.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Extract all fields from this referral form and return your answer as a JSON object.'}, {'type': 'image', 'image': 'images/gp-referral-cancer-colorectal IOV108_page_1.png'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '{"PatientSurname": {"value": "T108", "bbox": [0.043954, 0.218127, 0.233253, 0.240445]}, "PatientForeName": {"value": "T108", "bbox": [0.239541, 0.220393, 0.436699, 0.241599]}, "DateOfBirth": {"value": "1/5/61", "bbox": [0.04081, 0.242711, 0.28688, 0.265028]}, "Gender": {"value": "fEMale", "bbox": [0.296312, 0.240445, 0.507678, 0.263916]}, "Ethnicity": {"value": "Caucasian", "bbox": [0.04081, 0.26165, 0.27896, 0.285079]}, "Address": {"value": 

3. Supervised fine-tuning (SFT) with vision collator

Unsloth provides a vision data collator + standard SFTTrainer for this.

In [19]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))


CUDA available: True
CUDA device count: 1
Current device: 0
Device name: NVIDIA RTX A6000


### Training model

In [20]:
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch



# ---------------------------------------------------------
# 1) Load base model (Qwen3-VL-8B) in 4-bit
# ---------------------------------------------------------
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
    dtype = None,                      # auto dtype
    use_gradient_checkpointing = "unsloth"
)

print("Model loaded.")


# ---------------------------------------------------------
# 2) Add LoRA adapters
# ---------------------------------------------------------
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    target_modules = "all-linear",
    use_rslora  = False,              # Qwen3-VL does NOT benefit from RSLoRA
    loftq_config= None,
)


FastVisionModel.for_training(model)
print("LoRA enabled.")

param_device = next(model.parameters()).device
print("Model is on:", param_device)




# ---------------------------------------------------------
# 3) Vision-aware data collator
# ---------------------------------------------------------
data_collator = UnslothVisionDataCollator(model, tokenizer)


# ---------------------------------------------------------
# 4) Training config (SFT)
# ---------------------------------------------------------
train_args = SFTConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate              = 2e-4,
    max_steps                  = 300,      # <= ~3 epochs
    warmup_ratio               = 0.03,
    logging_steps              = 10,
    save_steps                 = 200,
    weight_decay               = 0.01,
    bf16                       = is_bf16_supported(),
    output_dir                 = "qwen3-vl-8b-referral-forms",
    remove_unused_columns      = False,
    max_seq_length             = 2048,
    dataset_text_field         = "",
    dataset_kwargs             = {"skip_prepare_dataset": True},
)


# ---------------------------------------------------------
# 5) Trainer
# ---------------------------------------------------------
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    args          = train_args,
    train_dataset = converted_ds,       # IMPORTANT: must be a Python list
    data_collator = data_collator,
)

print("Training…")
trainer.train()

print("Training complete.")


==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████████████| 2/2 [00:02<00:00,  1.33s/it]


Model loaded.


The model is already on multiple devices. Skipping the move to device specified in `args`.


LoRA enabled.
Model is on: cuda:0
Unsloth: Model does not have a default image size - using 512
Training…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 221 | Num Epochs = 4 | Total steps = 180
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.888000
20,0.660600
30,0.563500
40,0.518700
50,0.481500
60,0.468300
70,0.445800
80,0.449600
90,0.424400
100,0.415100


Training complete.


In [21]:
!nvidia-smi
print("Model device:", next(model.parameters()).device)

Sun Dec  7 00:26:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.03              Driver Version: 575.64.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:01:00.0 Off |                  Off |
| 30%   26C    P8             29W /  300W |   16235MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

This will fine-tune only the LoRA adapters while keeping Qwen3-VL’s vision + language backbone frozen, in 4-bit, which is what lets this run on modest GPUs.

4. Inference on a new referral form

Once training is done:

In [29]:
from PIL import Image
import torch

FastVisionModel.for_inference(model)

def run_inference(image_path, instruction):
    image = Image.open(image_path).convert("RGB")

    # Build a single-turn vision chat
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": instruction},
            ],
        }
    ]

    # Qwen3-VL template is already inside tokenizer from Unsloth 
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,
    )

    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 2048,
            temperature     = 0.2,
            top_p           = 0.8,
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

test_json = run_inference(
    "test.png",
    "Extract the referral form fields and return STRICT valid JSON only."
)
print(test_json)


user
Extract the referral form fields and return STRICT valid JSON only.
assistant
{"PatientName": {"value": "Ms Green", "type": "text", "bbox": [0.127822, 0.230459, 0.495519, 0.246058]}, "ReferringClinicianName": {"value": "Dr Good", "type": "text", "bbox": [0.514123, 0.230459, 0.903999, 0.247169]}, "NHSNumber": {"value": "01", "type": "text", "bbox": [0.127822, 0.247169, 0.495519, 0.26388]}, "ReferringClinicianAddress": {"value": "Birmingham", "type": "text", "bbox": [0.514123, 0.249392, 0.903999, 0.265036]}, "DateOfBirth": {"value": "6/1/71", "type": "text", "bbox": [0.127822, 0.26388, 0.286124, 0.277214]}, "Age": {"value": "54", "type": "text", "bbox": [0.394192, 0.26388, 0.490303, 0.279479]}, "Gender": {"value": "female", "type": "text", "bbox": [0.127822, 0.279479, 0.495519, 0.293924]}, "Address": {"value": "5 Timbuktu Street", "type": "text", "bbox": [0.127822, 0.295078, 0.495519, 0.313556]}, "Telephone": {"value": null, "type": "text", "bbox": [0.127822, 0.317299, 0.495519, 0.3

You can then json.loads(test_json) and validate / post-process.

5. Saving the fine-tuned Qwen3-VL model
Save LoRA adapters

In [ ]:
# adapter_dir = "qwen3-vl-8b-referral-forms-lora"

# model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
# print("Saved LoRA adapters to", adapter_dir)

(Optional) Merge LoRA into full weights for deployment

In [26]:
merged_dir = "qwen3-vl-8b-referral-forms-merged"

# merged = FastVisionModel.merge_and_unload(model)
model.save_pretrained_merged(merged_dir, tokenizer)
tokenizer.save_pretrained(merged_dir)

print("Saved merged model to", merged_dir)


Found HuggingFace hub cache directory: /home/Ubuntu/.cache/huggingface/hub


Fetching 1 files: 100%|███████████████████████████| 1/1 [00:00<00:00,  2.73it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██| 4/4 [00:50<00:00, 12.54s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|████████| 4/4 [01:29<00:00, 22.43s/it]


Unsloth: Merge process complete. Saved to `/home/Ubuntu/finetuning-qwen/qwen3-vl-8b-referral-forms-merged`
Saved merged model to qwen3-vl-8b-referral-forms-merged


In [28]:
model.push_to_hub_merged("sophy/qwen3-vl-8b-referral-forms", tokenizer, token="")

Processing Files (0 / 0): |                        |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|████████████████████| 11.4MB / 11.4MB, 28.5MB/s  
New Data Upload: |                                 |  0.00B /  0.00B,  0.00B/s  


Found HuggingFace hub cache directory: /home/Ubuntu/.cache/huggingface/hub


Fetching 1 files: 100%|███████████████████████████| 1/1 [00:00<00:00,  2.96it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██| 4/4 [00:47<00:00, 11.81s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Processing Files (0 / 0): |                        |  0.00B /  0.00B            
Processing Files (0 / 1):  25%|█████               | 1.25GB / 4.90GB,  178MB/s  
Processing Files (0 / 1):  25%|█████               | 1.25GB / 4.90GB,  174MB/s  
Processing Files (0 / 1):  26%|█████▏              | 1.26GB / 4.90GB,  171MB/s  
Processing Files (0 / 1):  26%|█████▏              | 1.27GB / 4.90GB,  167MB/s  
Processing Files (0 / 1):  26%|█████▏              | 1.28GB / 4.90GB,  164MB/s  
Processing Files (0 / 1):  27%|█████▎              | 1.30GB / 4.90GB,  163MB/s  
Processing Files (0 / 1):  27%|█████▍              | 1.33GB / 4.90GB,  162MB/s  
Processing Files (0 / 1):  28%|█████▌              | 1.35GB / 4.90GB,  161MB/s  
Processing Files (0 / 1):  28%|█████▌              | 1.37GB / 4.90GB,  160MB/s  
Processing Files (0 / 1):  29%|█████▋              | 1.40GB / 4.90GB,  159MB/s  
Processing Files (0 / 1):  29%|█████▊              | 1.41GB / 4.90GB,  157MB/s  
Processing Files (0 / 1):  2

Unsloth: Merge process complete. Saved to `/home/Ubuntu/finetuning-qwen/sophy/qwen3-vl-8b-referral-forms`


In [ ]:
# Save to 16bit GGUF

# model.save_pretrained_gguf("unsloth_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16")


In [ ]:
# model.push_to_hub_gguf("hf/unsloth_finetune_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16", token = "")

The merged model can be served with vLLM, llama.cpp (via GGUF exports), Docker, etc. – the same infra Unsloth shows for Qwen3-VL.

### Testing trained HF Model

In [31]:
from unsloth import FastVisionModel

MODEL_REPO = "sophy/qwen3-referral"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_REPO,
    load_in_4bit = True,
)

FastVisionModel.for_inference(model)


==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████████████| 4/4 [00:45<00:00, 11.32s/it]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear4bit(in_features=1152, out_features=3456, bias=True)
            (proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear4bit(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear4bit(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
   

In [34]:
from PIL import Image
import torch

image_path = "test.png"
image = Image.open(image_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Extract all fields and return JSON."},
        ],
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)

inputs = tokenizer(image, prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.1,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


user
Extract all fields and return JSON.
assistant
{"PatientName": {"value": "Ms Green", "bbox": [0.13129, 0.23001, 0.495519, 0.247722]}, "ReferringClinicianName": {"value": "Dr Good", "bbox": [0.514443, 0.23001, 0.891999, 0.250011]}, "NHSNumber": {"value": "01", "bbox": [0.128178, 0.247722, 0.495519, 0.264316]}, "ReferringClinicianAddress": {"value": "Birmingham", "bbox": [0.514443, 0.250011, 0.891999, 0.267723]}, "DateOfBirth": {"value": "6/1/71", "bbox": [0.128178, 0.264316, 0.340112, 0.279999]}, "Age": {"value": "54", "bbox": [0.392511, 0.264316, 0.495519, 0.281028]}, "Gender": {"value": "female", "bbox": [0.128178, 0.281028, 0.495519, 0.294118]}, "Address": {"value": "5 Timbuktu Street", "bbox": [0.128178, 0.296711, 0.495519, 0.314423]}, "Telephone": {"value": null, "bbox": [0.128178, 0.320003, 0.495519, 0.334308]}, "MobileNumber": {"value": null, "bbox": [0.128178, 0.337715, 0.495519, 0.353398]}, "Email": {"value": null, "bbox": [0.128178, 0.355427, 0.495519, 0.373139]}, "Consent

### GGUF

In [18]:
# BASE_MODEL   = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
MODEL_REPO = "sophy/finetuned-qwen-referrals"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_REPO,
    load_in_4bit = True,
    trust_remote_code = True
)



Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_Vl does not support SDPA - switching to fast eager.


Loading checkpoint shards: 100%|██████████████████| 4/4 [00:19<00:00,  4.77s/it]


In [22]:
import os
os.environ["HF_HUB_OFFLINE"]= "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"

In [ ]:
model.save_pretrained_gguf(
    "qwen3vl_referrals_gguf",
    tokenizer= tokenizer,
    quantization_method="q4_k_m"
)